In [10]:
from openai import AsyncOpenAI
from typing import Dict
import os
from pydantic import BaseModel
from pypdf import PdfReader
import requests # needed to do the push notification
import asyncio
from dotenv import load_dotenv
from agents import Agent
load_dotenv(override=True)


True

In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Activity(BaseModel):
    type: str
    piece_name: Optional[str] = None
    key: Optional[str] = None
    section: Optional[str] = None
    bars: Optional[str] = None
    focus: Optional[str] = None
    notes: Optional[str] = None

class PracticeSession(BaseModel):
    date: Optional[str]
    duration_min: Optional[int]
    activities: List[Activity]

In [62]:
PARSER_PROMPT = """
Extract structured practice data from the input.

Return a JSON object with EXACTLY this structure:

{
  "date": string or null,
  "duration_min": integer or null,
  "activities": [
    {
      "type": "warmup" | "piece" | "improvisation" | "sight_reading",
      "piece_name": string or null,
      "key": string or null,
      "section": string or null,
      "bars": string or null,
      "focus": string or null,
      "notes": string or null
    }
  ]
}

Rules:
- If date is missing, return null
- Convert durations like "~45 min" to integer
- "type" must be one of: warmup, piece, improvisation, sight_reading
- Do NOT invent fields
- If a field is not present, set it to null
- Do NOT normalize names (keep raw text)

Output rules:
- Return ONLY valid JSON
- Do NOT wrap in markdown or code fences
- Do NOT include explanations or extra text
- Output must start with "{" and end with "}"
"""

In [60]:
VALID_TYPES = ["warmup", "piece", "improvisation", "sight_reading"]
NORMALIZER_PROMPT = """
Normalize the structured practice session.

Rules:
- Standardize activity types to: warmup, piece, improvisation, sight_reading
- Normalize musical keys (e.g., "D major" → "D Major")
- Normalize piece names (fix spelling, standard classical naming)
- Do not remove information
- Keep bars/sections as-is

Return ONLY valid JSON.
- Do NOT wrap the JSON in code fences.
- Do NOT add explanations, comments, or markdown.
- Output must begin with "{" and end with "}".

"""

## Validators

In [5]:
from datetime import datetime

def validate_session(session: PracticeSession):
    if session.date is None:
        session.date = datetime.now().strftime("%Y-%m-%d")
    return session

def validate_types(session):
    for act in session.activities:
        if act.type not in VALID_TYPES:
            act.type = "other"
    return session

In [8]:
import sqlite3

conn = sqlite3.connect("piano.db")
cursor = conn.cursor()

def log_session(session, raw_text):
    cursor.execute(
        "INSERT INTO sessions (date, duration_min, raw_text) VALUES (?, ?, ?)",
        (session["date"], session["duration_min"], raw_text)
    )
    session_id = cursor.lastrowid

    for act in session["activities"]:
        composer_id = get_or_create_composer(act.get("composer"))
        piece_id = get_or_create_piece(act.get("piece_name"), composer_id)

        cursor.execute("""
            INSERT INTO activities (
                session_id, type,
                piece_id, composer_id,
                piece_name, composer_name,
                key, section, bars, focus, notes
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            session_id,
            act.get("type"),
            piece_id,
            composer_id,
            act.get("piece_name"),
            act.get("composer"),
            act.get("key"),
            act.get("section"),
            act.get("bars"),
            act.get("focus"),
            act.get("notes"),
        ))

    conn.commit()

### Pipeline orchestation

In [ ]:
def process_entry(raw_text: str):
    parsed = parser_agent.run(raw_text)
    parsed = PracticeSession(**parsed)

    normalized = normalizer_agent.run(parsed.model_dump())
    normalized = PracticeSession(**normalized)

    normalized = validate_types(normalized)

    log_session(normalized, raw_text)

### Agent definition

In [11]:
from agents import Agent
from openai import OpenAI
from agents import trace, Runner, OpenAIChatCompletionsModel


In [12]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
DeepSeek API Key exists and begins sk-


In [63]:
GEMINI_BASE_URL   = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"

deepseek_client   = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client     = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

deepseek_model    = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model      = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)

In [64]:
from typing import Dict, Any

parserAgent = Agent(name="parser_Agent", instructions=PARSER_PROMPT, model=deepseek_model)
normAgent   = Agent(name="Ge_adv_Agent", instructions=PARSER_PROMPT, model=gemini_model)
adv_agent3  = Agent(name="OAP_adv_Agent",instructions=PARSER_PROMPT,model="gpt-4o-mini")

In [65]:
message = f"April 14, ~45 min. Warmup in D major scales. Worked on Chopin Fantasie impromptyu, mostly middle section, struggling with left hand balance and memory. Some improv in A minor."
with trace("parser_Agent"):
    result = await Runner.run(parserAgent, message)
    print(result.final_output)

{
  "date": "April 14",
  "duration_min": 45,
  "activities": [
    {
      "type": "warmup",
      "piece_name": null,
      "key": "D major",
      "section": null,
      "bars": null,
      "focus": "scales",
      "notes": null
    },
    {
      "type": "piece",
      "piece_name": "Chopin Fantasie impromptyu",
      "key": null,
      "section": "middle section",
      "bars": null,
      "focus": "left hand balance and memory",
      "notes": "mostly middle section, struggling with left hand balance and memory"
    },
    {
      "type": "improvisation",
      "piece_name": null,
      "key": "A minor",
      "section": null,
      "bars": null,
      "focus": null,
      "notes": null
    }
  ]
}


In [74]:
import json
import re

def extract_json(text):
    if not text:
        raise ValueError("Empty output from agent")

    # Extract JSON block if wrapped in markdown
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in output: {text}")

    return json.loads(match.group())

JSONresult = extract_json(result.final_output)
print(JSONresult)
print(type(JSONresult))

{'date': 'April 14', 'duration_min': 45, 'activities': [{'type': 'warmup', 'piece_name': None, 'key': 'D major', 'section': None, 'bars': None, 'focus': 'scales', 'notes': None}, {'type': 'piece', 'piece_name': 'Chopin Fantasie impromptyu', 'key': None, 'section': 'middle section', 'bars': None, 'focus': 'left hand balance and memory', 'notes': 'mostly middle section, struggling with left hand balance and memory'}, {'type': 'improvisation', 'piece_name': None, 'key': 'A minor', 'section': None, 'bars': None, 'focus': None, 'notes': None}]}
<class 'dict'>


In [75]:
parsed = PracticeSession(**JSONresult)

In [80]:
import json

with trace("Norm"):
    normResult = await Runner.run(
        normAgent,
        json.dumps(parsed.model_dump())
    )

print(normResult.final_output)


```json
{"date": "April 14", "duration_min": 45, "activities": [{"type": "warmup", "piece_name": null, "key": "D major", "section": null, "bars": null, "focus": "scales", "notes": null}, {"type": "piece", "piece_name": "Chopin Fantasie impromptyu", "key": null, "section": "middle section", "bars": null, "focus": "left hand balance and memory", "notes": "mostly middle section, struggling with left hand balance and memory"}, {"type": "improvisation", "piece_name": null, "key": "A minor", "section": null, "bars": null, "focus": null, "notes": null}]}
```


In [81]:
JSONnorm = extract_json(normResult.final_output)
print(JSONnorm)


{'date': 'April 14', 'duration_min': 45, 'activities': [{'type': 'warmup', 'piece_name': None, 'key': 'D major', 'section': None, 'bars': None, 'focus': 'scales', 'notes': None}, {'type': 'piece', 'piece_name': 'Chopin Fantasie impromptyu', 'key': None, 'section': 'middle section', 'bars': None, 'focus': 'left hand balance and memory', 'notes': 'mostly middle section, struggling with left hand balance and memory'}, {'type': 'improvisation', 'piece_name': None, 'key': 'A minor', 'section': None, 'bars': None, 'focus': None, 'notes': None}]}


In [82]:
normalized = PracticeSession(**JSONnorm)